## Mục tiêu:
### 1. Concise 
- Là dùng API có sẵn của framework thay vì code chay từ đầu

### 2. Mapping kiến thức từ buổi 11
- Model: `net(X)` -> nn.LazyLinear(1)
- Loss: MSE -> `nn.MSELoss()`
- Optimizer: `sgd()` -> `torch.optim.SGD()`
- Training loop: vòng for -> `Trainer`

---
NOTE:
- reset gradient để batch sau không dùng gradient của batch trước 
- 

In [3]:
# code with concise

import torch
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


$$\hat y = XW + b$$

> [!NOTE] ELI5  
Phần Data là bước “tự chế” một bộ dữ liệu giả để mô hình học.  
Ta tự chọn công thức đúng trước, rồi thêm nhiễu nhỏ để dữ liệu giống thực tế hơn.  
Sau đó dùng dữ liệu này để xem model có học lại được công thức ban đầu hay không.

Cell Data của bạn là:

```python
true_w = torch.tensor([2, -3.4], device=device)
true_b = 4.2

n = 1000
X = torch.randn(n, 2, device=device)
print(X)
noise = 0.01 * torch.randn(n, device=device)
y = X @ true_w.reshape(-1, 1) + true_b + noise.reshape(-1, 1)
print(y)
```

Giải thích từng phần:

1. `true_w = torch.tensor([2, -3.4], device=device)`  
Đây là **weight thật** của bài toán.  
Nó nói rằng đầu ra $y$ được tạo từ 2 feature với mức ảnh hưởng:
- feature 1 nhân với $2$
- feature 2 nhân với $-3.4$

Về bản chất, đây là “đáp án gốc” mà lát nữa model phải học lại.

2. `true_b = 4.2`  
Đây là **bias thật**.

Công thức affine đầy đủ là:
$$
y = Xw + b
$$

Bias là phần cộng thêm cố định, độc lập với input.  
Nếu không có bias, đường hồi quy bị ép đi qua gốc trong nhiều cách biểu diễn.

3. `n = 1000`  
Số lượng mẫu dữ liệu.

Tức là bạn sẽ tạo ra 1000 điểm dữ liệu để train.  
$n$ càng lớn thì dữ liệu càng ổn định, mô hình càng dễ nhìn ra quy luật thật.

4. `X = torch.randn(n, 2, device=device)`  
Tạo ma trận input:
$$
X \in \mathbb{R}^{1000 \times 2}
$$

Ý nghĩa:
- có `1000` hàng, tức 1000 samples
- mỗi sample có `2` features

`torch.randn(...)` lấy số ngẫu nhiên từ phân phối chuẩn:
$$
X_{ij} \sim \mathcal{N}(0,1)
$$

Tại sao dùng Gaussian?
- dễ sinh dữ liệu
- giá trị phân bố quanh 0, khá “trung tính”
- rất thường dùng trong ví dụ học máy cơ bản

Ví dụ một dòng của `X` có thể là:
$$
x_i = [0.5,\,-1.2]
$$

5. `print(X)`  
Chỉ để xem dữ liệu đầu vào đã được tạo ra như thế nào.  
Về mặt học máy, dòng này không ảnh hưởng đến training.

6. `noise = 0.01 * torch.randn(n, device=device)`  
Tạo **nhiễu ngẫu nhiên** cho từng mẫu.

Ban đầu:
$$
\epsilon \sim \mathcal{N}(0,1)
$$

Sau khi nhân `0.01`:
$$
\epsilon \sim \mathcal{N}(0, 0.01^2)
$$

Tức là:
- trung bình bằng 0
- lúc dương lúc âm
- đa số rất nhỏ

Tác dụng:
- làm dữ liệu bớt hoàn hảo
- mô phỏng sai số đo lường ngoài đời
- giúp bài toán giống thực tế hơn

Nếu không có noise, toàn bộ điểm sẽ nằm đúng trên một siêu phẳng tuyến tính, quá sạch và quá lý tưởng.

7. `y = X @ true_w.reshape(-1, 1) + true_b + noise.reshape(-1, 1)`  
Đây là bước tạo **label** theo công thức tuyến tính có nhiễu.

Tách từng phần:

- `true_w.reshape(-1, 1)`  
`true_w` ban đầu có shape `(2,)`.  
Reshape thành `(2,1)` để nhân ma trận với `X`.

- `X @ true_w.reshape(-1, 1)`  
Đây là **matrix multiplication**:
$$
(1000 \times 2) @ (2 \times 1) = (1000 \times 1)
$$

Kết quả là mỗi sample được biến thành một số duy nhất:
$$
x_i^\top w
$$

Ví dụ nếu:
$$
x_i = [x_{i1}, x_{i2}], \quad w = [2, -3.4]
$$
thì:
$$
x_i^\top w = 2x_{i1} - 3.4x_{i2}
$$

- `+ true_b`  
Cộng bias 4.2 vào mọi sample.

- `+ noise.reshape(-1, 1)`  
Cộng thêm nhiễu riêng cho từng sample.  
`noise` ban đầu shape `(1000,)`, reshape thành `(1000,1)` để khớp với shape của `y`.

Toàn bộ phương trình là:
$$
y = Xw + b + \epsilon
$$

Đây chính là mô hình sinh dữ liệu chuẩn của linear regression.

8. `print(y)`  
In ra label vừa tạo.  
Cũng chỉ để quan sát, không ảnh hưởng logic.

Tóm lại, cell Data đang làm 3 việc:

1. Chọn quy luật thật:
$$
w = [2, -3.4], \quad b = 4.2
$$

2. Sinh input ngẫu nhiên:
$$
X \in \mathbb{R}^{1000 \times 2}
$$

3. Tạo output theo quy luật tuyến tính có nhiễu:
$$
y = Xw + b + \epsilon
$$

Ý nghĩa học thuật của cell này là: bạn đang tạo một dataset mà **ground truth đã biết trước**. Nhờ vậy, sau khi train xong, bạn có thể kiểm tra rất rõ:
- model có học ra gần đúng `true_w` không
- model có học ra gần đúng `true_b` không

Nếu muốn, tôi có thể viết tiếp cho bạn một phần giải thích ngay dưới cell này theo kiểu “comment từng dòng” để dán thẳng vào notebook Buoi 12.

In [5]:
# Data

true_w = torch.tensor([2, -3.4], device=device)
true_b = 4.2

n = 1000
X = torch.randn(n, 2, device=device)
print(X)
noise = 0.01 * torch.randn(n, device=device)
y = X @ true_w.reshape(-1, 1) + true_b + noise.reshape(-1, 1)
print(y)

tensor([[ 0.8303,  0.7305],
        [-1.3717,  0.5005],
        [ 1.8459,  1.5993],
        ...,
        [-1.7786, -1.5399],
        [ 1.3221,  0.2746],
        [ 1.5050,  1.3182]], device='cuda:0')
tensor([[ 3.3755e+00],
        [-2.4460e-01],
        [ 2.4421e+00],
        [ 2.1846e+00],
        [ 2.4934e+00],
        [ 5.4925e+00],
        [ 1.1325e+01],
        [ 1.0687e+01],
        [ 6.5593e+00],
        [ 1.1157e+01],
        [ 6.0116e+00],
        [ 2.5957e+00],
        [-2.1217e-01],
        [ 6.0999e+00],
        [ 1.2575e+01],
        [ 4.0962e+00],
        [ 9.7027e+00],
        [ 5.0869e+00],
        [-2.5500e+00],
        [ 5.1909e+00],
        [ 3.0355e+00],
        [-3.6729e+00],
        [ 1.2575e+00],
        [ 2.7460e+00],
        [ 4.7778e+00],
        [-4.5522e+00],
        [ 1.3348e+00],
        [ 4.7697e+00],
        [ 4.1835e+00],
        [ 2.7699e+00],
        [-3.3505e+00],
        [ 5.5637e+00],
        [ 2.8447e+00],
        [ 7.6499e+00],
        [ 2.3374e+0

In [6]:
# Dataloader
batch_size = 32
dataset = torch.utils.data.TensorDataset(X, y)
loader = torch.utils.data.DataLoader(dataset, batch_size, shuffle=True)

In [8]:
# Model
net = nn.LazyLinear(1, device=device)
net.weight.data.normal_(0, 0.01)
net.bias.data.fill_(0)

tensor([], device='cuda:0')

> [!NOTE] ELI5  
Cell model là nơi bạn định nghĩa “cái máy dự đoán”.  
Với linear regression, cái máy này rất đơn giản: lấy input, nhân với weight, rồi cộng bias.  
Nếu input có 2 đặc trưng, model sẽ học xem mỗi đặc trưng nên ảnh hưởng mạnh hay yếu đến kết quả.

Nếu cell model của bạn có dạng như một trong hai kiểu này:

```python
net = nn.LazyLinear(1)
```

hoặc

```python
net = nn.Sequential(nn.LazyLinear(1))
```

thì bản chất là như nhau. Giải thích như sau.

## 1. `nn.LazyLinear(1)` là gì?

Đây là một **fully connected layer** có 1 đầu ra.

Công thức toán:
$$
\hat y = XW + b
$$

Trong đó:
- $X$: input
- $W$: weight model phải học
- $b$: bias model phải học
- $\hat y$: giá trị dự đoán

Vì bạn đang làm **linear regression 1 output**, nên `1` ở đây nghĩa là:
- mỗi sample đi vào
- model trả ra đúng 1 số dự đoán

Ví dụ:
- input một sample có 2 feature: `[x_1, x_2]`
- output là 1 giá trị:
$$
\hat y = w_1x_1 + w_2x_2 + b
$$

## 2. Tại sao là `LazyLinear` chứ không phải `Linear(2, 1)`?

`LazyLinear(1)` nghĩa là:
- bạn nói trước: “tôi muốn 1 output”
- còn số chiều input bao nhiêu thì PyTorch tự suy ra ở lần forward đầu tiên

Ví dụ khi bạn đưa `X` có shape `(1000, 2)` vào:
- PyTorch nhìn thấy input có 2 features
- nó tự tạo weight shape `(1, 2)`
- bias shape `(1,)`

Nói cách khác:
- `nn.Linear(2, 1)` = bạn tự chỉ rõ input có 2 chiều
- `nn.LazyLinear(1)` = framework tự phát hiện input có 2 chiều

Dùng `LazyLinear` tiện khi:
- chưa muốn hard-code số chiều input
- muốn code ngắn hơn
- demo nhanh trong notebook

## 3. Model này học cái gì?

Model không học dữ liệu `X`.  
Model học **tham số**:
- weight
- bias

Nếu dữ liệu được tạo từ:
$$
y = 2x_1 - 3.4x_2 + 4.2 + \epsilon
$$

thì model sẽ cố học sao cho:
- weight gần `[2, -3.4]`
- bias gần `4.2`

Tức là nó tìm lại quy luật ẩn phía sau dữ liệu.

## 4. Nếu có `nn.Sequential(...)` thì nghĩa là gì?

Ví dụ:

```python
net = nn.Sequential(nn.LazyLinear(1))
```

`nn.Sequential` chỉ là một “cái hộp” chứa các layer theo thứ tự.

Trong ví dụ này chỉ có đúng 1 layer bên trong, nên nó tương đương với:

```python
net = nn.LazyLinear(1)
```

Khác biệt chủ yếu là về cách tổ chức code:
- `LazyLinear(1)` trực tiếp: gọn hơn
- `Sequential(...)`: tiện khi sau này muốn xếp nhiều layer liên tiếp

Ví dụ mạng sâu hơn:
```python
net = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)
```

Nhưng với linear regression cơ bản thì chỉ cần 1 layer linear là đủ.

## 5. Tại sao nói đây vẫn là linear regression?

Vì cell model này vẫn chỉ làm đúng một phép biến đổi affine:
$$
\hat y = XW + b
$$

Chưa có:
- activation phi tuyến như `ReLU`, `Sigmoid`, `Tanh`
- nhiều tầng sâu
- attention hay convolution

Nên bản chất toán học vẫn là linear regression, chỉ khác là:
- buổi 11: tự viết `X @ w + b`
- buổi 12: dùng API có sẵn `nn.LazyLinear(1)`

## 6. Weight và bias trong cell model nằm ở đâu?

Sau khi model được khởi tạo và chạy qua một lần forward, bạn có thể xem:

```python
net.weight
net.bias
```

Ý nghĩa:
- `net.weight`: ma trận trọng số
- `net.bias`: độ lệch cộng thêm

Nếu input có 2 features và output có 1 giá trị thì thường:
- `weight` có shape `(1, 2)`
- `bias` có shape `(1,)`

## 7. Nếu model cell có thêm phần khởi tạo weight nhỏ Gaussian

Thường bạn sẽ thấy:

```python
net.weight.data.normal_(0, 0.01)
net.bias.data.fill_(0)
```

Điều này nghĩa là:
- weight khởi tạo ngẫu nhiên nhỏ quanh 0
- bias khởi tạo bằng 0

Mục tiêu:
- bắt đầu từ điểm trung tính
- giúp quá trình học ổn định hơn

## 8. Tóm gọn cell model

Cell model đang nói:

1. Tạo một hàm dự đoán có dạng
$$
\hat y = XW + b
$$
2. Cho PyTorch quản lý `W` và `b`
3. Để sau này `backward()` tự tính gradient và optimizer tự cập nhật

Nếu muốn, tôi có thể viết luôn cho bạn một đoạn giải thích ngắn để chèn trực tiếp dưới cell model trong notebook Buoi 12.

In [9]:
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.03)

In [10]:
epochs = 5
for epoch in range(epochs):
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        l = criterion(net(X_batch), y_batch)
        l.backward()
        optimizer.step()
    with torch.no_grad():
        train_l = criterion(net(X), y)
        print(f"Epoch {epoch + 1}, loss {train_l:f}")
        

Epoch 1, loss 0.696167
Epoch 2, loss 0.015818
Epoch 3, loss 0.000468
Epoch 4, loss 0.000110
Epoch 5, loss 0.000100


In [11]:
w_hat = net.weight.data.reshape(-1)
b_hat = net.bias.data.item()
print("w that:", true_w)
print("w hoc duoc:", w_hat)
print("b that:", true_b)
print("b hoc duoc:", b_hat)

w that: tensor([ 2.0000, -3.4000], device='cuda:0')
w hoc duoc: tensor([ 2.0003, -3.4004], device='cuda:0')
b that: 4.2
b hoc duoc: 4.199690341949463
